# Swing+ Results

**2024-2025 Leaderboard : Top Swing+**

In [37]:
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / 'data').exists())
DATA = ROOT / 'data'

MIN_SWINGS = 300   # qualify: >= 300 competitive swings in 2024-25 (2026 is the held-out test season)

# Swing+ = a batter's mean xrv_grade (per-swing xRV z-scored to a 0-100 scale, 50 = league-avg swing)
sw = pd.read_parquet(DATA / 'xrv_swings.parquet',
                     columns=['batter_id', 'game_year', 'xrv', 'xrv_grade'])
names = (pd.read_parquet(DATA / 'swings_model.parquet', columns=['batter_id', 'batter_full_name'])
           .drop_duplicates('batter_id'))

board = (sw[sw.game_year.isin([2024, 2025])]
         .groupby('batter_id')
         .agg(swings=('xrv_grade', 'size'),
              swing_plus=('xrv_grade', 'mean'),
              mean_xrv=('xrv', 'mean'))
         .reset_index()
         .query('swings >= @MIN_SWINGS')
         .merge(names, on='batter_id', how='left')
         .sort_values('swing_plus', ascending=False)
         .reset_index(drop=True))
board.index += 1
print(f'Qualified batters (>= {MIN_SWINGS} swings, 2024-25): {len(board)}')

top = board[['batter_full_name', 'swings', 'swing_plus', 'mean_xrv']].head(25)
top.columns = ['Batter', 'Swings', 'Swing+', 'mean xRV']
top.round({'Swing+': 1, 'mean xRV': 4})

Qualified batters (>= 300 swings, 2024-25): 508


,Batter,Swings,Swing+,mean xRV
1,Patrick Wisdom,319,54.6,0.0071
2,Juan Soto,2004,53.8,0.0017
3,Matt Wallner,1166,53.4,-0.0007
4,Nick Kurtz,767,53.2,-0.0018
5,Christopher Morel,1584,52.9,-0.0037
6,Matt Mervis,321,52.7,-0.0055
7,Garrett Mitchell,497,52.7,-0.0055
8,Shohei Ohtani,2741,52.5,-0.0062
9,Kyle Schwarber,2219,52.5,-0.0068
10,Heriberto Hernández,481,52.4,-0.0073


**2024-2025 Leaderboard : Top Swing+ by swing-shape cluster**

In [38]:
# Top Swing+ by swing-shape cluster (per-(batter, stand) GMM; cluster 0 = the hitter's primary swing).
# Clusters are per-hitter and NOT comparable across hitters -- each row is one hitter's own shape.
# Archetype = Layer-1 league label + its situational tag (cards.py), so a bare "cluster 2" reads as
# e.g. "Uppercut Pull · down, vs offspd" -- two same-archetype shapes separate by their situation.
MIN_CLUSTER_SWINGS = 100

swp = pd.read_parquet(DATA / 'xrv_swings.parquet', columns=['play_id', 'game_year', 'xrv', 'xrv_grade'])
ca = pd.read_parquet(DATA / 'cluster_assignments.parquet',
                     columns=['play_id', 'batter_id', 'batter_stand', 'cluster'])
rep = pd.read_parquet(DATA / 'batter_repertoire.parquet',
                      columns=['batter_id', 'batter_stand', 'label'])
cards = pd.read_parquet(DATA / 'shape_cards.parquet',
                        columns=['batter_id', 'batter_stand', 'cluster', 'archetype_detailed'])

by_cluster = (ca.merge(swp, on='play_id', how='inner')
              .query('game_year in [2024, 2025]')
              .groupby(['batter_id', 'batter_stand', 'cluster'])
              .agg(swings=('xrv_grade', 'size'),
                   swing_plus=('xrv_grade', 'mean'),
                   mean_xrv=('xrv', 'mean'))
              .reset_index()
              .query('swings >= @MIN_CLUSTER_SWINGS')
              .merge(rep, on=['batter_id', 'batter_stand'], how='left')
              .merge(cards, on=['batter_id', 'batter_stand', 'cluster'], how='left')
              .sort_values('swing_plus', ascending=False)
              .reset_index(drop=True))
by_cluster.index += 1
print(f'Qualified (batter, cluster) shapes (>= {MIN_CLUSTER_SWINGS} swings, 2024-25): {len(by_cluster)}')

top_c = by_cluster[['label', 'cluster', 'archetype_detailed', 'swings', 'swing_plus', 'mean_xrv']].head(25)
top_c.columns = ['Batter', 'Cluster', 'Archetype (· situation)', 'Swings', 'Swing+', 'mean xRV']
top_c.round({'Swing+': 1, 'mean xRV': 4})

Qualified (batter, cluster) shapes (>= 100 swings, 2024-25): 1380


,Batter,Cluster,Archetype (· situation),Swings,Swing+,mean xRV
1,Colson Montgomery,0,Primary,161,60.5,0.0443
2,Ryan McMahon,0,Primary,690,59.2,0.0358
3,Luis Robert Jr.,0,Primary,473,58.4,0.0313
4,Cal Raleigh (L),1,"Uppercut Pull · 3-2, ahead, down",518,58.4,0.0307
5,Patrick Wisdom,0,Primary,207,58.2,0.0298
6,Jake Bauers,0,Primary,405,58.2,0.0298
7,Shohei Ohtani,1,"Level Oppo · inside, vs FB, 0-0",750,58.2,0.0295
8,Christian Yelich,0,Primary,530,57.6,0.0258
9,Pete Alonso,1,"Uppercut Pull · down, inside",585,57.4,0.0244
10,Heriberto Hernández,0,Primary,241,57.4,0.0244


In [39]:
def cluster_table(batter_name):
    """Return a table of the batter's swing clusters (archetype + situational tag), sorted by Swing+."""
    d = by_cluster[by_cluster['label'].str.contains(batter_name, case=False, na=False)]
    return (d[['label', 'cluster', 'archetype_detailed', 'swings', 'swing_plus', 'mean_xrv']]
            .sort_values('swing_plus', ascending=False)
            .reset_index(drop=True)
            .rename(columns={'label': 'Batter', 'cluster': 'Cluster', 'archetype_detailed': 'Archetype (· situation)',
                             'swings': 'Swings', 'swing_plus': 'Swing+', 'mean_xrv': 'mean xRV'})
            .round({'Swing+': 1, 'mean xRV': 4}))

cluster_table('Raleigh')

,Batter,Cluster,Archetype (· situation),Swings,Swing+,mean xRV
0,Cal Raleigh (L),1,"Uppercut Pull · 3-2, ahead, down",518,58.4,0.0307
1,Cal Raleigh (L),0,Primary,546,54.4,0.0055
2,Cal Raleigh (R),0,Primary,392,53.8,0.0018
3,Cal Raleigh (R),1,"Uppercut Pull · down, vs offspd, inside",363,48.3,-0.0335
4,Cal Raleigh (L),3,"Level Oppo · up, vs FB, away",268,46.5,-0.0446
5,Cal Raleigh (L),4,"Uppercut Pull · down, vs offspd, away",131,42.9,-0.0677
6,Cal Raleigh (L),2,"Uppercut Pull · down, vs offspd, inside",350,40.3,-0.0843


**Swing Repertoire**

In [36]:
# Swing Repertoire leaderboard: Repertoire+ (repertoire expansiveness -- geometry only, how WIDE a
# hitter's swing-shape profile is) with each unit's Swing+ (value) merged in. The two are independent
# axes: Repertoire+ = breadth of shapes, Swing+ = mean quality of those swings.
# Repertoire+ is sorted high->low; lead with repertoire_pctile (0-100 rank) since single-shape units
# pile up at the 0-spread floor and skew the "100 = average" scale.
repertoire = pd.read_parquet(DATA / 'repertoire_scores.parquet',
                             columns=['batter_id', 'batter_stand', 'label', 'k',
                                      'repertoire_plus', 'repertoire_pctile'])

# unit-level Swing+ = mean xrv_grade over the unit's 2024-25 swings, keyed per (batter_id, batter_stand)
swp = pd.read_parquet(DATA / 'xrv_swings.parquet', columns=['play_id', 'game_year', 'xrv_grade'])
ca = pd.read_parquet(DATA / 'cluster_assignments.parquet',
                     columns=['play_id', 'batter_id', 'batter_stand'])
unit_swing_plus = (ca.merge(swp, on='play_id', how='inner')
                   .query('game_year in [2024, 2025]')
                   .groupby(['batter_id', 'batter_stand'])['xrv_grade']
                   .mean().rename('swing_plus').reset_index())

board = (repertoire.merge(unit_swing_plus, on=['batter_id', 'batter_stand'], how='left')
         .sort_values('repertoire_plus', ascending=False)
         .reset_index(drop=True))
board.index += 1
print(f'Qualified (batter, stand) units: {len(board)}')

lead = board[['label', 'batter_stand', 'k', 'repertoire_plus', 'swing_plus']].head(25)
lead.columns = ['Batter', 'R/L', 'Shapes (k)', 'Repertoire+', 'Swing+']
lead.round({'Repertoire+': 1, 'Swing+': 1})

Qualified (batter, stand) units: 703


,Batter,R/L,Shapes (k),Repertoire+,Swing+
1,Johnathan Rodríguez,R,2,114.3,48.9
2,Michael Toglia (L),L,2,114.1,49.7
3,Spencer Jones,L,2,112.8,47.0
4,Bo Bichette,R,2,112.3,48.3
5,Rece Hinds,R,2,111.6,48.8
6,Bobby Dalbec,R,2,111.4,50.2
7,Nathan Church,L,2,111.2,48.0
8,Tyler Freeman,R,2,111.1,50.1
9,Heliot Ramos,R,2,111.1,50.1
10,Jake Cronenworth,L,2,111.1,49.3
